In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC, SVR
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.ensemble import GradientBoostingClassifier, GradientBoostingRegressor
from sklearn.ensemble import AdaBoostClassifier, AdaBoostRegressor
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor, plot_tree
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler
from pathlib import Path
import joblib

DATA_DIR = Path('../data')
MODEL_DIR = Path('../model')
MODEL_DIR.mkdir(exist_ok=True)

In [2]:
df = pd.read_csv(DATA_DIR / 'Breast Cancer Wisconsin.csv')

In [3]:
df.head()

,id,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,...,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst,Unnamed: 32
0,842302,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,...,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,NaN
1,842517,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,...,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,NaN
2,84300903,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,...,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,NaN
3,84348301,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,...,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300,NaN
4,84358402,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,...,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678,NaN


In [4]:
df.dtypes

id                           int64
diagnosis                      str
radius_mean                float64
texture_mean               float64
perimeter_mean             float64
area_mean                  float64
smoothness_mean            float64
compactness_mean           float64
concavity_mean             float64
concave points_mean        float64
symmetry_mean              float64
fractal_dimension_mean     float64
radius_se                  float64
texture_se                 float64
perimeter_se               float64
area_se                    float64
smoothness_se              float64
compactness_se             float64
concavity_se               float64
concave points_se          float64
symmetry_se                float64
fractal_dimension_se       float64
radius_worst               float64
texture_worst              float64
perimeter_worst            float64
area_worst                 float64
smoothness_worst           float64
compactness_worst          float64
concavity_worst     

In [5]:
df.shape

(569, 33)

In [6]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 569 entries, 0 to 568
Data columns (total 33 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   id                       569 non-null    int64  
 1   diagnosis                569 non-null    str    
 2   radius_mean              569 non-null    float64
 3   texture_mean             569 non-null    float64
 4   perimeter_mean           569 non-null    float64
 5   area_mean                569 non-null    float64
 6   smoothness_mean          569 non-null    float64
 7   compactness_mean         569 non-null    float64
 8   concavity_mean           569 non-null    float64
 9   concave points_mean      569 non-null    float64
 10  symmetry_mean            569 non-null    float64
 11  fractal_dimension_mean   569 non-null    float64
 12  radius_se                569 non-null    float64
 13  texture_se               569 non-null    float64
 14  perimeter_se             569 non-null

In [7]:
df.describe().T

,count,mean,std,min,25%,50%,75%,max
id,569.0,3.037183e+07,1.250206e+08,8670.000000,869218.000000,906024.000000,8.813129e+06,9.113205e+08
radius_mean,569.0,1.412729e+01,3.524049e+00,6.981000,11.700000,13.370000,1.578000e+01,2.811000e+01
texture_mean,569.0,1.928965e+01,4.301036e+00,9.710000,16.170000,18.840000,2.180000e+01,3.928000e+01
perimeter_mean,569.0,9.196903e+01,2.429898e+01,43.790000,75.170000,86.240000,1.041000e+02,1.885000e+02
area_mean,569.0,6.548891e+02,3.519141e+02,143.500000,420.300000,551.100000,7.827000e+02,2.501000e+03
smoothness_mean,569.0,9.636028e-02,1.406413e-02,0.052630,0.086370,0.095870,1.053000e-01,1.634000e-01
compactness_mean,569.0,1.043410e-01,5.281276e-02,0.019380,0.064920,0.092630,1.304000e-01,3.454000e-01
concavity_mean,569.0,8.879932e-02,7.971981e-02,0.000000,0.029560,0.061540,1.307000e-01,4.268000e-01
concave points_mean,569.0,4.891915e-02,3.880284e-02,0.000000,0.020310,0.033500,7.400000e-02,2.012000e-01
symmetry_mean,569.0,1.811619e-01,2.741428e-02,0.106000,0.161900,0.179200,1.957000e-01,3.040000e-01


In [8]:
df = df.drop(columns=['Unnamed: 32'], errors='ignore')

In [9]:
df.head()

,id,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,...,radius_worst,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst
0,842302,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,...,25.38,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890
1,842517,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,...,24.99,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902
2,84300903,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,...,23.57,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758
3,84348301,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,...,14.91,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300
4,84358402,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,...,22.54,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678


In [10]:
df['diagnosis'].value_counts()
df['diagnosis'].value_counts(normalize=True) * 100

diagnosis
B    62.741652
M    37.258348
Name: proportion, dtype: float64

In [11]:
df.drop(columns=['id', 'diagnosis']).corr()


,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,symmetry_mean,fractal_dimension_mean,...,radius_worst,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst
radius_mean,1.000000,0.323782,0.997855,0.987357,0.170581,0.506124,0.676764,0.822529,0.147741,-0.311631,...,0.969539,0.297008,0.965137,0.941082,0.119616,0.413463,0.526911,0.744214,0.163953,0.007066
texture_mean,0.323782,1.000000,0.329533,0.321086,-0.023389,0.236702,0.302418,0.293464,0.071401,-0.076437,...,0.352573,0.912045,0.358040,0.343546,0.077503,0.277830,0.301025,0.295316,0.105008,0.119205
perimeter_mean,0.997855,0.329533,1.000000,0.986507,0.207278,0.556936,0.716136,0.850977,0.183027,-0.261477,...,0.969476,0.303038,0.970387,0.941550,0.150549,0.455774,0.563879,0.771241,0.189115,0.051019
area_mean,0.987357,0.321086,0.986507,1.000000,0.177028,0.498502,0.685983,0.823269,0.151293,-0.283110,...,0.962746,0.287489,0.959120,0.959213,0.123523,0.390410,0.512606,0.722017,0.143570,0.003738
smoothness_mean,0.170581,-0.023389,0.207278,0.177028,1.000000,0.659123,0.521984,0.553695,0.557775,0.584792,...,0.213120,0.036072,0.238853,0.206718,0.805324,0.472468,0.434926,0.503053,0.394309,0.499316
compactness_mean,0.506124,0.236702,0.556936,0.498502,0.659123,1.000000,0.883121,0.831135,0.602641,0.565369,...,0.535315,0.248133,0.590210,0.509604,0.565541,0.865809,0.816275,0.815573,0.510223,0.687382
concavity_mean,0.676764,0.302418,0.716136,0.685983,0.521984,0.883121,1.000000,0.921391,0.500667,0.336783,...,0.688236,0.299879,0.729565,0.675987,0.448822,0.754968,0.884103,0.861323,0.409464,0.514930
concave points_mean,0.822529,0.293464,0.850977,0.823269,0.553695,0.831135,0.921391,1.000000,0.462497,0.166917,...,0.830318,0.292752,0.855923,0.809630,0.452753,0.667454,0.752399,0.910155,0.375744,0.368661
symmetry_mean,0.147741,0.071401,0.183027,0.151293,0.557775,0.602641,0.500667,0.462497,1.000000,0.479921,...,0.185728,0.090651,0.219169,0.177193,0.426675,0.473200,0.433721,0.430297,0.699826,0.438413
fractal_dimension_mean,-0.311631,-0.076437,-0.261477,-0.283110,0.584792,0.565369,0.336783,0.166917,0.479921,1.000000,...,-0.253691,-0.051269,-0.205151,-0.231854,0.504942,0.458798,0.346234,0.175325,0.334019,0.767297


In [12]:
df['diagnosis'].map({'B': 0, 'M': 1}).value_counts()


diagnosis
0    357
1    212
Name: count, dtype: int64

In [13]:
neon_colors = ["#00FFFF", "#FF00FF"]
plt.style.use("dark_background")
plt.figure(figsize=(8, 8))
counts = df['diagnosis'].value_counts()
plt.pie(counts, labels=['Benign (B)', 'Malignant (M)'], autopct='%1.1f%%', 
        startangle=90, colors=neon_colors, pctdistance=0.85, 
        textprops={'color':"white", 'weight':'bold'})

centre_circle = plt.Circle((0,0),0.70,fc='black')
fig = plt.gcf()
fig.gca().add_artist(centre_circle)

plt.title('Target Distribution: Benign vs Malignant', color='white', weight='bold')
plt.show()

C:\Users\CoRe.2.Duo\AppData\Local\Temp\ipykernel_25228\4200988359.py:14: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [14]:
plt.figure(figsize=(12, 10))
mean_cols = [c for c in df.columns if 'mean' in c]
corr_matrix = df[mean_cols].corr()

sns.heatmap(corr_matrix, annot=True, fmt=".1f", cmap='cool', 
            linewidths=0.5, linecolor='black', cbar_kws={'label': 'Correlation Coefficient'})

plt.title('Feature Correlation Matrix (Mean Features)', color='white', weight='bold', pad=20)
plt.xticks(rotation=45, ha='right')
plt.show()

C:\Users\CoRe.2.Duo\AppData\Local\Temp\ipykernel_25228\2691158191.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [15]:
data_norm = (df[mean_cols] - df[mean_cols].mean()) / df[mean_cols].std()
data_norm = pd.concat([df['diagnosis'], data_norm], axis=1)
data_melted = pd.melt(data_norm, id_vars="diagnosis", var_name="features", value_name="value")

plt.figure(figsize=(16, 8))
sns.violinplot(x="features", y="value", hue="diagnosis", data=data_melted, 
               split=True, inner="quart", palette=neon_colors)

plt.xticks(rotation=45, color='white')
plt.title('Feature Distribution Comparison (Standardized)', color='white', weight='bold')
plt.xlabel('Features')
plt.ylabel('Standardized Values')
plt.legend(title='Diagnosis', loc='upper right')
plt.show()

C:\Users\CoRe.2.Duo\AppData\Local\Temp\ipykernel_25228\1145682407.py:14: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [16]:
plt.figure(figsize=(10, 8))
sns.scatterplot(data=df, x='radius_mean', y='texture_mean', hue='diagnosis', 
                palette=neon_colors, s=100, alpha=0.8, edgecolor='white')

plt.title('Cluster Separation: Radius vs Texture', color='white', weight='bold')
plt.xlabel('Mean Radius')
plt.ylabel('Mean Texture')
plt.legend(title='Diagnosis')
plt.grid(color='gray', linestyle='--', linewidth=0.3, alpha=0.5)
plt.show()

C:\Users\CoRe.2.Duo\AppData\Local\Temp\ipykernel_25228\80671454.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [17]:
worst_features = ['radius_worst', 'perimeter_worst', 'area_worst', 'compactness_worst']

# Melting data for seaborn
df_melt_worst = pd.melt(df, id_vars=['diagnosis'], value_vars=worst_features)

plt.figure(figsize=(14, 8))
sns.boxenplot(x='variable', y='value', hue='diagnosis', data=df_melt_worst, 
              palette=neon_colors, k_depth='trustworthy')

plt.title('Critical "Worst" Features Analysis (Boxen Plot)', color='white', weight='bold', fontsize=16)
plt.xlabel('Clinical Features (Worst Case)')
plt.ylabel('Measured Values')
plt.yscale('log') # Log scale because "Area" is much larger than others
plt.legend(title='Diagnosis', loc='upper right')
plt.show()

C:\Users\CoRe.2.Duo\AppData\Local\Temp\ipykernel_25228\727952675.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [18]:
plt.figure(figsize=(12, 6))
sns.histplot(data=df, x='radius_mean', hue='diagnosis', multiple="stack", palette=neon_colors, bins=30)

plt.title('Distribution of Tumor Radius', color='white', pad=20)
plt.xlabel('Mean Radius')
plt.ylabel('Number of Patients')
plt.grid(axis='y', linestyle='--', alpha=0.3)
plt.show()

C:\Users\CoRe.2.Duo\AppData\Local\Temp\ipykernel_25228\3219869766.py:8: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [19]:
plt.figure(figsize=(10, 6))
sns.histplot(data=df, x='area_mean', hue='diagnosis', 
             kde=True, bins=40, palette=neon_colors, 
             alpha=0.5, edgecolor='black', linewidth=0.5)

plt.title('Distribution of Tumor Area (with KDE Curve)', color='white', pad=20, fontsize=16)
plt.xlabel('Tumor Area (Mean)', labelpad=10)
plt.ylabel('Frequency (Count)', labelpad=10)

plt.axvline(x=700, color='yellow', linestyle='--', linewidth=1.5, alpha=0.8)
plt.text(720, 30, 'Visual Separation (~700)', color='yellow', fontsize=10)

plt.grid(axis='y', linestyle=':', alpha=0.2)
plt.legend(title='Diagnosis', labels=['Malignant (M)', 'Benign (B)'], loc='upper right')
plt.show()

C:\Users\CoRe.2.Duo\AppData\Local\Temp\ipykernel_25228\2737668795.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [20]:
# FIX: 'id' is a row identifier, not a feature, so drop it together with the target
X = df.drop(columns=['id', 'diagnosis'])
y = df['diagnosis']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [21]:
# Optional model leaderboard (pip install lazypredict). Skipped if not installed.
try:
    from lazypredict.Supervised import LazyClassifier
    clf = LazyClassifier(verbose=0, ignore_warnings=True, custom_metric=None)
    models, predictions = clf.fit(X_train, X_test, y_train, y_test)
    print(models.head(5))
except ImportError:
    print('lazypredict not installed - skipping leaderboard')

lazypredict not installed - skipping leaderboard


In [22]:
svm_model = SVC(kernel='linear', random_state=42)
svm_model.fit(X_train_scaled, y_train)
svm_pred = svm_model.predict(X_test_scaled)
svm_acc = accuracy_score(y_test, svm_pred)

In [23]:
dt_model = DecisionTreeClassifier(random_state=42, max_depth=5)
dt_model.fit(X_train_scaled, y_train) 
dt_pred = dt_model.predict(X_test_scaled)
dt_acc = accuracy_score(y_test, dt_pred)

In [24]:
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train_scaled, y_train)
rf_pred = rf_model.predict(X_test_scaled)
rf_acc = accuracy_score(y_test, rf_pred)

In [25]:
gb_model = GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, random_state=42)
gb_model.fit(X_train_scaled, y_train)
gb_pred = gb_model.predict(X_test_scaled)
gb_acc = accuracy_score(y_test, gb_pred)

In [26]:
print(f"SVM Accuracy: {svm_acc:.4f}")
print(f"Decision Tree Accuracy: {dt_acc:.4f}")
print(f"Random Forest Accuracy: {rf_acc:.4f}")
print(f"Gradient Boosting Accuracy: {gb_acc:.4f}")

SVM Accuracy: 0.9561
Decision Tree Accuracy: 0.9474
Random Forest Accuracy: 0.9649
Gradient Boosting Accuracy: 0.9561


## Train & save the deployed model

The API (`app/`) uses the 10 `*_mean` features so a person can type them into the web form. Same split (`random_state=42`) as above. Saves `model/breast_cancer_model.pkl` and `model/scaler.pkl`, which `app/model.py` loads.

The scaler is fitted on a **DataFrame** so it remembers the column names/order; the API sends a DataFrame with the same names.

In [27]:
FEATURES = ['radius_mean', 'texture_mean', 'perimeter_mean', 'area_mean',
            'smoothness_mean', 'compactness_mean', 'concavity_mean',
            'concave points_mean', 'symmetry_mean', 'fractal_dimension_mean']

Xd_train, Xd_test, yd_train, yd_test = train_test_split(df[FEATURES], y, test_size=0.2, random_state=42)
deploy_scaler = StandardScaler().fit(Xd_train)
deploy_model = RandomForestClassifier(n_estimators=100, random_state=42)
deploy_model.fit(deploy_scaler.transform(Xd_train), yd_train)

pred = deploy_model.predict(deploy_scaler.transform(Xd_test))
print('classes_ (predict_proba column order):', deploy_model.classes_)
print(f'Held-out accuracy (10 features): {accuracy_score(yd_test, pred):.4f}')
print(classification_report(yd_test, pred))

classes_ (predict_proba column order): ['B' 'M']
Held-out accuracy (10 features): 0.9561
              precision    recall  f1-score   support

           B       0.96      0.97      0.97        71
           M       0.95      0.93      0.94        43

    accuracy                           0.96       114
   macro avg       0.96      0.95      0.95       114
weighted avg       0.96      0.96      0.96       114



In [28]:
joblib.dump(deploy_model, MODEL_DIR / 'breast_cancer_model.pkl')
joblib.dump(deploy_scaler, MODEL_DIR / 'scaler.pkl')
print('Saved to', MODEL_DIR.resolve())

Saved to E:\project ml deployment\model
